In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

In [10]:
raw_path = Path("../data/raw")
csv_files = list(raw_path.glob("*.csv"))
print("CSV files found:")
for file in csv_files:
    print(file)

file_path = csv_files[0]
df = pd.read_csv(file_path)
print("\nDataset loaded successfully!")
print("Shape:", df.shape)

CSV files found:
..\data\raw\transaction_data.csv

Dataset loaded successfully!
Shape: (6362620, 11)


In [11]:
df.head()



,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [12]:
print("Column names:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)

Column names: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']

Data types:
step                int64
type                  str
amount            float64
nameOrig              str
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest              str
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64
dtype: object


In [13]:
missing = df.isnull().sum()
print("Missing values per column:")
print(missing)
print("Total missing values:", missing.sum())

duplicates = df.duplicated().sum()
print("\nDuplicate rows:", duplicates)

Missing values per column:
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64
Total missing values: 0

Duplicate rows: 0


In [14]:
fraud_count = df["isFraud"].sum()
total_transactions = len(df)
fraud_rate = fraud_count / total_transactions * 100

print("Fraud Statistics:")
print(f"Total transactions: {total_transactions:,}")
print(f"Fraud transactions: {fraud_count:,}")
print(f"Legitimate transactions: {total_transactions - fraud_count:,}")
print(f"Fraud rate: {fraud_rate:.4f}%")

Fraud Statistics:
Total transactions: 6,362,620
Fraud transactions: 8,213
Legitimate transactions: 6,354,407
Fraud rate: 0.1291%


In [15]:
fraud_by_type = df[df["isFraud"] == 1]["type"].value_counts()
print("Fraud transactions by type:")
print(fraud_by_type)
print("\n(Sanity check: should ONLY be TRANSFER and CASH_OUT)")

Fraud transactions by type:
type
CASH_OUT    4116
TRANSFER    4097
Name: count, dtype: int64

(Sanity check: should ONLY be TRANSFER and CASH_OUT)


In [16]:
min_step = df["step"].min()
max_step = df["step"].max()
total_steps = max_step - min_step + 1

print("Minimum step:", min_step)
print("Maximum step:", max_step)
print("Total steps:", total_steps)

Minimum step: 1
Maximum step: 743
Total steps: 743


In [17]:
DELAY_DAYS = 7
DELAY_STEPS = DELAY_DAYS * 24

print(f"Delay window: {DELAY_DAYS} days ({DELAY_STEPS} steps)")
print("Total available steps:", total_steps)
print(f"Steps remaining after a {DELAY_DAYS}-day window:", total_steps - DELAY_STEPS)

Delay window: 7 days (168 steps)
Total available steps: 743
Steps remaining after a 7-day window: 575


In [18]:
df["day_bucket"] = (df["step"] - df["step"].min()) // 24
fraud_per_day = df[df["isFraud"] == 1].groupby("day_bucket").size()
print("Fraud cases per day:")
print(fraud_per_day)

df["week_bucket"] = (df["step"] - df["step"].min()) // (24 * 7)
fraud_per_week = df[df["isFraud"] == 1].groupby("week_bucket").size()
print("\nFraud cases per week (note: last week is partial, not a real drop):")
print(fraud_per_week)

Fraud cases per day:
day_bucket
0     271
1     309
2     310
3     262
4     252
5     228
6     272
7     278
8     255
9     282
10    262
11    298
12    242
13    246
14    250
15    252
16    320
17    268
18    256
19    236
20    272
21    256
22    216
23    280
24    240
25    272
26    280
27    248
28    260
29    268
30    272
dtype: int64

Fraud cases per week (note: last week is partial, not a real drop):
week_bucket
0    1904
1    1863
2    1854
3    1792
4     800
dtype: int64


In [19]:
fraud_df = df[df["isFraud"] == 1]

print("Fraud by origin account (repeat offenders):")
print(fraud_df["nameOrig"].value_counts().head(10))

print("\nFraud by destination account (repeat targets):")
print(fraud_df["nameDest"].value_counts().head(10))

print("\n--- OVERALL ACCOUNT REUSE (not just fraud subset) ---")
print(f"Unique nameOrig: {df['nameOrig'].nunique():,}")
print(f"nameOrig appearing more than once: {(df['nameOrig'].value_counts() > 1).sum():,}")
print(f"Unique nameDest: {df['nameDest'].nunique():,}")
print(f"nameDest appearing more than once: {(df['nameDest'].value_counts() > 1).sum():,}")

Fraud by origin account (repeat offenders):
nameOrig
C1305486145    1
C840083671     1
C1420196421    1
C2101527076    1
C137533655     1
C1118430673    1
C749981943     1
C1334405552    1
C467632528     1
C1364127192    1
Name: count, dtype: int64

Fraud by destination account (repeat targets):
nameDest
C410033330     2
C803116137     2
C904300960     2
C1013511446    2
C2020337583    2
C200064275     2
C185805228     2
C52390890      2
C935310781     2
C1827219533    2
Name: count, dtype: int64

--- OVERALL ACCOUNT REUSE (not just fraud subset) ---
Unique nameOrig: 6,353,307
nameOrig appearing more than once: 9,298
Unique nameDest: 2,722,362
nameDest appearing more than once: 459,658


In [20]:
print("isFlaggedFraud value counts:")
print(df["isFlaggedFraud"].value_counts())
print("\nFraud vs isFlaggedFraud:")
print(pd.crosstab(df["isFraud"], df["isFlaggedFraud"], margins=True))

isFlaggedFraud value counts:


NameError: name 'df' is not defined

### Check `isFlaggedFraud`

`isFlaggedFraud` is an existing fraud indicator in the dataset.

- `isFraud` → actual fraud / **target label**
- `isFlaggedFraud` → existing fraud flag / possible feature

Only **16 transactions** are flagged, while there are **8,213 actual frauds**. Therefore, `isFlaggedFraud` has very limited usefulness as a fraud signal.


### Prepare Data for Time-Based Features

Create a timestamp from the hourly `step` column and sort transactions by destination account and time for rolling calculations.

In [21]:
df["timestamp"] = pd.to_datetime(df["step"], unit="h", origin="2023-01-01")
print("Timestamp column created.")
print(df[["step", "timestamp"]].head())

Timestamp column created.
   step           timestamp
0     1 2023-01-01 01:00:00
1     1 2023-01-01 01:00:00
2     1 2023-01-01 01:00:00
3     1 2023-01-01 01:00:00
4     1 2023-01-01 01:00:00


In [22]:
df = df.reset_index().rename(columns={"index": "original_order"})
df = df.sort_values(["nameDest", "timestamp"])
print("Sorted for rolling computation.")
print(f"Rows: {len(df):,}")

Sorted for rolling computation.
Rows: 6,362,620


### Destination Transaction History

Calculate the number of previous transactions for each destination account in the last 24 hours and 7 days, excluding the current transaction to avoid data leakage.

In [23]:
grouped = df.set_index("timestamp").groupby("nameDest")["amount"]

df["destination_transactions_last_24h"] = (
    grouped.rolling("24h", closed="left").count().reset_index(drop=True)
)
df["destination_transactions_last_7d"] = (
    grouped.rolling("168h", closed="left").count().reset_index(drop=True)
)

print("Destination rolling counts computed.")
print(df[["nameDest", "timestamp", "destination_transactions_last_24h",
          "destination_transactions_last_7d"]].head(10))

Destination rolling counts computed.
            nameDest           timestamp  destination_transactions_last_24h  \
4987517  C1000004082 2023-01-15 16:00:00                                NaN   
5032095  C1000004082 2023-01-15 18:00:00                                NaN   
5219127  C1000004082 2023-01-16 10:00:00                                NaN   
5331822  C1000004082 2023-01-16 14:00:00                                NaN   
5462768  C1000004082 2023-01-16 19:00:00                                NaN   
5659867  C1000004082 2023-01-17 12:00:00                                NaN   
1148643  C1000004940 2023-01-06 11:00:00                                NaN   
1165745  C1000004940 2023-01-06 12:00:00                                NaN   
1762000  C1000004940 2023-01-07 17:00:00                                3.0   
2020611  C1000004940 2023-01-08 12:00:00                                1.0   

         destination_transactions_last_7d  
4987517                               NaN  
50320

### Destination Previous Average Amount

Calculate the average transaction amount previously received by each destination account and identify first-time transactions.

In [24]:
df["destination_avg_previous_amount"] = (
    df.groupby("nameDest")["amount"]
      .apply(lambda x: x.shift(1).expanding().mean())
      .reset_index(drop=True)
)

df["destination_is_first_transaction"] = df["destination_avg_previous_amount"].isna().astype(int)
df["destination_avg_previous_amount"] = df["destination_avg_previous_amount"].fillna(0)

print("Destination average previous amount computed.")
print(f"First-time destinations: {df['destination_is_first_transaction'].sum():,}")

Destination average previous amount computed.
First-time destinations: 2,722,362


### Destination Amount Deviation

Calculate how much the current transaction amount differs from the destination's previous average amount.

In [25]:
df["destination_amount_deviation"] = df["amount"] - df["destination_avg_previous_amount"]
print("Destination amount deviation computed.")
print(df["destination_amount_deviation"].describe())

Destination amount deviation computed.
count    6.362620e+06
mean     3.431261e+04
std      6.884530e+05
min     -3.967488e+07
25%     -1.354149e+05
50%      9.038280e+03
75%      1.219456e+05
max      9.233041e+07
Name: destination_amount_deviation, dtype: float64


In [26]:
df = df.sort_values("original_order").drop(columns="original_order").reset_index(drop=True)
print("Original row order restored.")

Original row order restored.


### Balance Consistency & Step-Level Features

Create balance error features for the origin and destination accounts. Also calculate the total transactions, total transaction amount, and average transaction amount for each hourly step.

In [27]:
df["origin_balance_error"] = df["oldbalanceOrg"] - df["amount"] - df["newbalanceOrig"]
df["destination_balance_error"] = df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"]

print("Balance consistency features created.")
print("\nOrigin balance error:")
print(df["origin_balance_error"].describe())
print("\nDestination balance error:")
print(df["destination_balance_error"].describe())

Balance consistency features created.

Origin balance error:
count    6.362620e+06
mean    -2.010925e+05
std      6.066505e+05
min     -9.244552e+07
25%     -2.496411e+05
50%     -6.867726e+04
75%     -2.954230e+03
max      1.000000e-02
Name: origin_balance_error, dtype: float64

Destination balance error:
count    6.362620e+06
mean     5.556717e+04
std      4.415288e+05
min     -7.588573e+07
25%      0.000000e+00
50%      3.500490e+03
75%      2.935305e+04
max      1.319123e+07
Name: destination_balance_error, dtype: float64


In [28]:
step_stats = (
    df.groupby("step")["amount"]
      .agg(total_transactions="count", total_transaction_amount="sum",
           avg_transaction_amount="mean")
      .reset_index()
)
df = df.merge(step_stats, on="step", how="left")
print("Step-level features created.")
print(df[["step", "total_transactions", "total_transaction_amount",
          "avg_transaction_amount"]].head())

Step-level features created.
   step  total_transactions  total_transaction_amount  avg_transaction_amount
0     1                2708              2.854292e+08           105402.208696
1     1                2708              2.854292e+08           105402.208696
2     1                2708              2.854292e+08           105402.208696
3     1                2708              2.854292e+08           105402.208696
4     1                2708              2.854292e+08           105402.208696


In [29]:
print("Correlation with isFraud:")
print(df[["origin_balance_error", "destination_balance_error",
          "destination_amount_deviation"]].corrwith(df["isFraud"]))

df["destination_balance_is_zero"] = (
    (df["oldbalanceDest"] == 0) & (df["newbalanceDest"] == 0)
).astype(int)

print("\nCrosstab: destination_balance_is_zero vs isFraud (row %)")
print(pd.crosstab(df["destination_balance_is_zero"], df["isFraud"], normalize="index"))

Correlation with isFraud:
origin_balance_error            0.011283
destination_balance_error       0.055120
destination_amount_deviation    0.070493
dtype: float64

Crosstab: destination_balance_is_zero vs isFraud (row %)
isFraud                             0         1
destination_balance_is_zero                    
0                            0.998977  0.001023
1                            0.998241  0.001759


destination_amount_deviation = 0.0705 means higher deviation has a slight positive relationship with fraud.

### Final Feature Plan

- **Original features used:** `amount`, `type`
- **Engineered features:** destination history, amount deviation, balance errors, zero-balance flag, and step-level statistics
- **Target label:** `isFraud`
- `isFlaggedFraud` was checked but not included due to very limited coverage.

In [30]:
print("FINAL FEATURE PLAN")
print("\nTransaction features: amount, type")
print("\nDestination history features:")
for f in ["destination_transactions_last_24h", "destination_transactions_last_7d",
          "destination_avg_previous_amount", "destination_amount_deviation",
          "destination_is_first_transaction"]:
    print("-", f)
print("\nBalance consistency features:")
for f in ["origin_balance_error", "destination_balance_error", "destination_balance_is_zero"]:
    print("-", f)
print("\nSystem-level features:")
for f in ["total_transactions", "total_transaction_amount", "avg_transaction_amount"]:
    print("-", f)
print("\nNote: account age and device/location flags dropped — PaySim's schema")
print("and near-total origin-account non-reuse don't support them.")

FINAL FEATURE PLAN

Transaction features: amount, type

Destination history features:
- destination_transactions_last_24h
- destination_transactions_last_7d
- destination_avg_previous_amount
- destination_amount_deviation
- destination_is_first_transaction

Balance consistency features:
- origin_balance_error
- destination_balance_error
- destination_balance_is_zero

System-level features:
- total_transactions
- total_transaction_amount
- avg_transaction_amount

Note: account age and device/location flags dropped — PaySim's schema
and near-total origin-account non-reuse don't support them.


In [31]:
# Save the fully feature-engineered dataframe to disk.
# Parquet is faster and smaller than CSV for this size of data.
df.to_parquet("../data/processed/paysim_features.parquet", index=False)
print("Saved processed data.")
print(f"Rows: {len(df):,}, Columns: {len(df.columns)}")

Saved processed data.
Rows: 6,362,620, Columns: 25


In [ ]:

df = pd.get_dummies(df, columns=["type"], prefix="type")
print("Encoded columns:")
print([c for c in df.columns if c.startswith("type_")])

Encoded columns:
['type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER']


In [ ]:


feature_columns = [
    "amount",
    "destination_transactions_last_24h",
    "destination_transactions_last_7d",
    "destination_avg_previous_amount",
    "destination_amount_deviation",
    "destination_is_first_transaction",
    "origin_balance_error",
    "destination_balance_error",
    "destination_balance_is_zero",
    "total_transactions",
    "total_transaction_amount",
    "avg_transaction_amount",
] + [c for c in df.columns if c.startswith("type_")]

target_column = "isFraud"

print(f"Using {len(feature_columns)} features.")
print(feature_columns)

Using 17 features.
['amount', 'destination_transactions_last_24h', 'destination_transactions_last_7d', 'destination_avg_previous_amount', 'destination_amount_deviation', 'destination_is_first_transaction', 'origin_balance_error', 'destination_balance_error', 'destination_balance_is_zero', 'total_transactions', 'total_transaction_amount', 'avg_transaction_amount', 'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER']


In [34]:
def get_period_data(weeks):
    """
    weeks: a list of week_bucket numbers to include, e.g. [0] or [0, 1]
    Returns X (features) and y (true fraud labels) for just those weeks.
    """
    subset = df[df["week_bucket"].isin(weeks)]
    X = subset[feature_columns]
    y = subset[target_column]
    return X, y

# Quick sanity check
X_week0, y_week0 = get_period_data([0])
print(f"Week 0: {len(X_week0):,} rows, {y_week0.sum():,} fraud cases")

Week 0: 1,930,180 rows, 1,904 fraud cases


In [ ]:
# Train on Week 0 only
X_train_initial, y_train_initial = get_period_data([0])


X_week1, y_week1 = get_period_data([1])
X_week2, y_week2 = get_period_data([2])
X_week3, y_week3 = get_period_data([3])

# Retrained version: Week 0 + Week 1 combined
X_train_retrained, y_train_retrained = get_period_data([0, 1])

print("Drift-decay datasets ready:")
print(f"Initial train (Week 0): {len(X_train_initial):,} rows")
print(f"Retrained train (Week 0+1): {len(X_train_retrained):,} rows")
print(f"Eval sets — Week1: {len(X_week1):,}, Week2: {len(X_week2):,}, Week3: {len(X_week3):,}")

Drift-decay datasets ready:
Initial train (Week 0): 1,930,180 rows
Retrained train (Week 0+1): 4,784,775 rows
Eval sets — Week1: 2,854,595, Week2: 1,279,261, Week3: 232,135


In [ ]:
X_naive_train, y_naive_train = get_period_data([0, 1, 2])

X_delay_train, y_delay_train = get_period_data([0, 1])

X_delay_test, y_delay_test = get_period_data([3])

print("Delay-aware comparison datasets ready:")
print(f"Naive train (Weeks 0-2): {len(X_naive_train):,} rows, {y_naive_train.sum():,} fraud")
print(f"Delay-aware train (Weeks 0-1): {len(X_delay_train):,} rows, {y_delay_train.sum():,} fraud")
print(f"Shared test (Week 3): {len(X_delay_test):,} rows, {y_delay_test.sum():,} fraud")

Delay-aware comparison datasets ready:
Naive train (Weeks 0-2): 6,064,036 rows, 5,621 fraud
Delay-aware train (Weeks 0-1): 4,784,775 rows, 3,767 fraud
Shared test (Week 3): 232,135 rows, 1,792 fraud


In [37]:
for wk in [0, 1, 2, 3]:
    X, y = get_period_data([wk])
    rate = y.sum() / len(y) * 100
    print(f"Week {wk}: {len(y):,} rows, {y.sum():,} fraud, rate = {rate:.4f}%")

Week 0: 1,930,180 rows, 1,904 fraud, rate = 0.0986%
Week 1: 2,854,595 rows, 1,863 fraud, rate = 0.0653%
Week 2: 1,279,261 rows, 1,854 fraud, rate = 0.1449%
Week 3: 232,135 rows, 1,792 fraud, rate = 0.7720%


In [38]:
print(df.groupby("day_bucket").size())

day_bucket
0     574255
1     455238
2       1070
3      28240
4       9789
5     441005
6     420583
7     449637
8     417919
9     392945
10    417859
11    349776
12    428583
13    397876
14    401282
15    375594
16    425766
17     20999
18     11300
19     19727
20     24593
21     53437
22     51012
23     32709
24     57853
25     13885
26      8578
27     14661
28     54890
29     11287
30       272
dtype: int64


In [40]:
full_daily_counts = df.groupby("day_bucket").size()
print("Days 0-15:")
print(full_daily_counts.iloc[0:16].to_string())
print("\nDays 16-30:")
print(full_daily_counts.iloc[16:31].to_string())

Days 0-15:
day_bucket
0     574255
1     455238
2       1070
3      28240
4       9789
5     441005
6     420583
7     449637
8     417919
9     392945
10    417859
11    349776
12    428583
13    397876
14    401282
15    375594

Days 16-30:
day_bucket
16    425766
17     20999
18     11300
19     19727
20     24593
21     53437
22     51012
23     32709
24     57853
25     13885
26      8578
27     14661
28     54890
29     11287
30       272


In [41]:
dense_window = df[(df["day_bucket"] >= 5) & (df["day_bucket"] <= 16)]
print(f"Dense window total: {len(dense_window):,} rows")
print(f"Dense window fraud: {dense_window['isFraud'].sum():,}")
print("\nFraud per day within dense window:")
print(dense_window[dense_window["isFraud"] == 1].groupby("day_bucket").size())

Dense window total: 4,918,825 rows
Dense window fraud: 3,185

Fraud per day within dense window:
day_bucket
5     228
6     272
7     278
8     255
9     282
10    262
11    298
12    242
13    246
14    250
15    252
16    320
dtype: int64


In [42]:
def get_dense_period(days):
    subset = dense_window[dense_window["day_bucket"].isin(days)]
    X = subset[feature_columns]
    y = subset[dense_window["isFraud"].name] if False else subset["isFraud"]
    return X, y

periods = {
    "train_initial": [5, 6, 7],
    "eval_1":        [8, 9, 10],
    "eval_2":        [11, 12, 13],
    "held_out":      [14, 15, 16],
}

for name, days in periods.items():
    X, y = get_dense_period(days)
    rate = y.sum() / len(y) * 100
    print(f"{name} (days {days}): {len(X):,} rows, {y.sum():,} fraud, rate = {rate:.4f}%")

train_initial (days [5, 6, 7]): 1,311,225 rows, 778 fraud, rate = 0.0593%
eval_1 (days [8, 9, 10]): 1,228,723 rows, 799 fraud, rate = 0.0650%
eval_2 (days [11, 12, 13]): 1,176,235 rows, 786 fraud, rate = 0.0668%
held_out (days [14, 15, 16]): 1,202,642 rows, 822 fraud, rate = 0.0683%


### Preparing Data for Baseline Modeling

Reload the processed data, recreate the time-based periods, select the dense window, and handle missing values before training the baseline model.

In [2]:
import pandas as pd

df = pd.read_parquet("../data/processed/paysim_features.parquet")
if "week_bucket" not in df.columns:
    df["week_bucket"] = (df["step"] - df["step"].min()) // (24 * 7)
if "day_bucket" not in df.columns:
    df["day_bucket"] = (df["step"] - df["step"].min()) // 24

feature_columns = [
    "amount",
    "destination_transactions_last_24h",
    "destination_transactions_last_7d",
    "destination_avg_previous_amount",
    "destination_amount_deviation",
    "destination_is_first_transaction",
    "origin_balance_error",
    "destination_balance_error",
    "destination_balance_is_zero",
    "total_transactions",
    "total_transaction_amount",
    "avg_transaction_amount",
] + [c for c in df.columns if c.startswith("type_")]

target_column = "isFraud"

dense_window = df[(df["day_bucket"] >= 5) & (df["day_bucket"] <= 16)]

def get_dense_period(days):
    subset = dense_window[dense_window["day_bucket"].isin(days)]
    return subset[feature_columns], subset["isFraud"]

print(f"Reloaded. {len(df):,} total rows, {len(dense_window):,} dense-window rows.")

Reloaded. 6,362,620 total rows, 4,918,825 dense-window rows.


In [ ]:


nan_check = df[feature_columns].isna().sum()
print("NaN count per feature (before fill):")
print(nan_check[nan_check > 0])

df["destination_transactions_last_24h"] = df["destination_transactions_last_24h"].fillna(0)
df["destination_transactions_last_7d"] = df["destination_transactions_last_7d"].fillna(0)

# Rebuild dense_window since it was sliced from df BEFORE this fix
dense_window = df[(df["day_bucket"] >= 5) & (df["day_bucket"] <= 16)]

# Confirm no NaN remains
nan_check_after = df[feature_columns].isna().sum()
print("\nNaN count per feature (after fill):")
print(nan_check_after[nan_check_after > 0] if nan_check_after.sum() > 0 else "None — all clean.")

NaN count per feature (before fill):
destination_transactions_last_24h    4008404
destination_transactions_last_7d     2876033
dtype: int64

NaN count per feature (after fill):
None — all clean.


### Baseline Logistic Regression

Train a Logistic Regression baseline on Days 5–7 using balanced class weights because fraud is highly imbalanced. Evaluate it on future Days 8–10 using PR-AUC as the performance metric.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_recall_curve

X_train, y_train = get_dense_period([5, 6, 7])

baseline_model = LogisticRegression(max_iter=1000, class_weight="balanced")
baseline_model.fit(X_train, y_train)

print("Baseline logistic regression trained.")
print(f"Trained on {len(X_train):,} rows, {y_train.sum():,} fraud cases.")

Baseline logistic regression trained.
Trained on 1,311,225 rows, 778 fraud cases.


In [7]:
X_eval1, y_eval1 = get_dense_period([8, 9, 10])
y_scores = baseline_model.predict_proba(X_eval1)[:, 1]
pr_auc = average_precision_score(y_eval1, y_scores)
print(f"Baseline PR-AUC on eval_1: {pr_auc:.4f}")

Baseline PR-AUC on eval_1: 0.5701
